In [1]:
import torch
import torch.nn as nn
import torch.optim as optim
from tqdm import tqdm
from torchvision import datasets, transforms, models
from torch.utils.data import DataLoader
import os


In [2]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

Using device: cpu


In [3]:
data_dir = r"D:\DEEPFAKE\datasets\video_dataset\train"

transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
])

In [4]:
full_dataset = datasets.ImageFolder(root=data_dir, transform=transform)

train_size = int(0.8 * len(full_dataset))
val_size = len(full_dataset) - train_size

train_dataset, val_dataset = torch.utils.data.random_split(full_dataset, [train_size, val_size])

train_loader = DataLoader(train_dataset, batch_size=8, shuffle=True, num_workers=0)
val_loader   = DataLoader(val_dataset, batch_size=8, shuffle=False, num_workers=0)

print("Classes:", full_dataset.classes)

Classes: ['fake', 'real']


In [5]:
from torchvision import models
from torchvision.models import EfficientNet_B0_Weights

In [6]:
model = models.efficientnet_b0(weights = EfficientNet_B0_Weights.DEFAULT)


In [7]:
layers = list(model.features.children())

freeze_until = int(0.7 * len(layers))

for i, layer in enumerate(layers):
    if i < freeze_until:
        for param in layer.parameters():
            param.requires_grad = False

In [8]:
num_features = model.classifier[1].in_features

model.classifier = nn.Sequential(
    nn.Dropout(0.3),
    nn.Linear(num_features, 2)   # 2 classes
)

model = model.to(device)

In [9]:
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=1e-4)

In [10]:
epochs = 4

print("🚀 Training Started...")

for epoch in range(epochs):
    model.train()
    running_loss = 0
    correct = 0
    total = 0

    print(f"\n🔥 Epoch {epoch+1}/{epochs}")

    for images, labels in tqdm(train_loader):
        images, labels = images.to(device), labels.to(device)

        optimizer.zero_grad()

        outputs = model(images)
        loss = criterion(outputs, labels)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

        _, preds = torch.max(outputs, 1)
        correct += (preds == labels).sum().item()
        total += labels.size(0)

    train_acc = correct / total
    train_loss = running_loss / len(train_loader)

    # ✅ Validation
    model.eval()
    val_correct = 0
    val_total = 0

    with torch.no_grad():
        for images, labels in tqdm(val_loader):
            images, labels = images.to(device), labels.to(device)

            outputs = model(images)
            _, preds = torch.max(outputs, 1)

            val_correct += (preds == labels).sum().item()
            val_total += labels.size(0)

    val_acc = val_correct / val_total

    print(f"✅ Train Loss: {train_loss:.4f} | Train Acc: {train_acc:.4f} | Val Acc: {val_acc:.4f}")

🚀 Training Started...

🔥 Epoch 1/4


100%|██████████| 57/57 [00:07<00:00,  7.18it/s]


✅ Train Loss: 0.2592 | Train Acc: 0.9092 | Val Acc: 0.9714

🔥 Epoch 2/4


100%|██████████| 57/57 [00:05<00:00, 10.51it/s]


✅ Train Loss: 0.1049 | Train Acc: 0.9620 | Val Acc: 0.9824

🔥 Epoch 3/4


100%|██████████| 57/57 [00:04<00:00, 11.51it/s]


✅ Train Loss: 0.0958 | Train Acc: 0.9626 | Val Acc: 0.9736

🔥 Epoch 4/4


100%|██████████| 57/57 [00:03<00:00, 14.61it/s]

✅ Train Loss: 0.0688 | Train Acc: 0.9780 | Val Acc: 0.9780


In [11]:
os.makedirs("AI_models/video", exist_ok=True)

torch.save(model.state_dict(), "AI_models/video/efficientnet_video_pytorch.pth")

print("🔥 PyTorch Video model saved successfully!")

🔥 PyTorch Video model saved successfully!
